# <center> Advanced Python Exam 2026 </center>

## Rules

1) The exam is open-book, thus you're allowed to have written notes, but you are still **not** allowed to carry and/or use flash drives and other\
electronic tech. and/or software of any kind to assist you

2) The exam will take 2 hours 40 minutes

3) You can ask any questions, but no answer is promised unless it is to a question related to the task description

4) Import the neccessary modules in the cell below the `Imports` label

If you're going to be crying please do it silently so as not to disturb the rest of the students

## Imports

In [ ]:
#your imports here

## Task 1 (50 pts). Find Books with No Available Copies

### Task Description

Table: `library_books`

| Column Name      | Type    |
|------------------|---------|
| book_id          | int     |
| title            | varchar |
| author           | varchar |
| genre            | varchar |
| publication_year | int     |
| total_copies     | int     |

book_id is the unique identifier for this table.\
Each row contains information about a book in the library, including the total number of copies owned by the library.\
Table: borrowing_records


| Column Name   | Type    |
|---------------|---------|
| record_id     | int     |
| book_id       | int     |
| borrower_name | varchar |
| borrow_date   | date    |
| return_date   | date    |

record_id is the unique identifier for this table.\
Each row represents a borrowing transaction and return_date is NULL if the book is currently borrowed and hasn't been returned yet.\
Write a solution to find all books that are currently borrowed (not returned) and have zero copies available in the library.

A book is considered currently borrowed if there exists a borrowing record with a NULL return_date\
Return the result table ordered by current borrowers in descending order, then by book title in ascending order.

The result format is in the following example. Wrap your solution in the `sol_1` func

 

Example:

Input:

library_books table:

| book_id | title                  | author           | genre    | publication_year | total_copies |
|---------|------------------------|------------------|----------|------------------|--------------|
| 1       | The Great Gatsby       | F. Scott         | Fiction  | 1925             | 3            |
| 2       | To Kill a Mockingbird  | Harper Lee       | Fiction  | 1960             | 3            |
| 3       | 1984                   | George Orwell    | Dystopian| 1949             | 1            |
| 4       | Pride and Prejudice    | Jane Austen      | Romance  | 1813             | 2            |
| 5       | The Catcher in the Rye | J.D. Salinger    | Fiction  | 1951             | 1            |
| 6       | Brave New World        | Aldous Huxley    | Dystopian| 1932             | 4            |

borrowing_records table:


| record_id | book_id | borrower_name | borrow_date | return_date |
|-----------|---------|---------------|-------------|-------------|
| 1         | 1       | Alice Smith   | 2024-01-15  | NULL        |
| 2         | 1       | Bob Johnson   | 2024-01-20  | NULL        |
| 3         | 2       | Carol White   | 2024-01-10  | 2024-01-25  |
| 4         | 3       | David Brown   | 2024-02-01  | NULL        |
| 5         | 4       | Emma Wilson   | 2024-01-05  | NULL        |
| 6         | 5       | Frank Davis   | 2024-01-18  | 2024-02-10  |
| 7         | 1       | Grace Miller  | 2024-02-05  | NULL        |
| 8         | 6       | Henry Taylor  | 2024-01-12  | NULL        |
| 9         | 2       | Ivan Clark    | 2024-02-12  | NULL        |
| 10        | 2       | Jane Adams    | 2024-02-15  | NULL        |

Output:


| book_id | title            | author        | genre     | publication_year | current_borrowers |
|---------|------------------|---------------|-----------|------------------|-------------------|
| 1       | The Great Gatsby | F. Scott      | Fiction   | 1925             | 3                 | 
| 3       | 1984             | George Orwell | Dystopian | 1949             | 1                 |

Explanation:

The Great Gatsby (book_id = 1):\
Total copies: 3\
Currently borrowed by Alice Smith, Bob Johnson, and Grace Miller (3 borrowers)\
Available copies: 3 - 3 = 0\
Included because available_copies = 0

1984 (book_id = 3):\
Total copies: 1\
Currently borrowed by David Brown (1 borrower)\
Available copies: 1 - 1 = 0\
Included because available_copies = 0

Books not included:\
To Kill a Mockingbird (book_id = 2): Total copies = 3, current borrowers = 2, available = 1\
Pride and Prejudice (book_id = 4): Total copies = 2, current borrowers = 1, available = 1\
The Catcher in the Rye (book_id = 5): Total copies = 1, current borrowers = 0, available = 1\
Brave New World (book_id = 6): Total copies = 4, current borrowers = 1, available = 3\

Result ordering:
The Great Gatsby appears first with 3 current borrowers\
1984 appears second with 1 current borrower\
Output table is ordered by current_borrowers in descending order, then by book_title in ascending order

### Solution

In [10]:
def sol_1(library_books, borrowing_records):
    df = borrowing_records.copy()
    df = df[df['return_date'].isna()][['book_id', 'borrower_name']].groupby('book_id').count().rename(columns={'borrower_name' : 'current_borrowers'}).reset_index().merge(library_books[['book_id', 'total_copies']], on='book_id')
    df = df[df['current_borrowers'] == df['total_copies']][['book_id', 'current_borrowers']]
    df = df.merge(library_books[['book_id', 'title', 'author', 'genre', 'publication_year']], on='book_id', how='left')
    return df.iloc[:, [0, *list(range(2,6)), 1]].sort_values(['current_borrowers', 'title'], ascending=[False, True])

### Tests

Use the code block below to test your solutions

In [12]:
from pandas.testing import assert_frame_equal

print(f"Running test cases...\n")
passed = 0
failed = 0
for i in range(1, 81):
    status = f"\rTest {i}....."
    expected = pd.read_csv(fr'tests_task_1\exp_{i}.csv', index_col=0)
    lib, bor = pd.read_csv(fr'tests_task_1\library_{i}.csv', index_col=0), pd.read_csv(fr'tests_task_1\borrowing_{i}.csv', index_col=0)
    result = sol_1(lib, bor)
    try:
        assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
        print(status+"PASS", end='')
        passed += 1
    except AssertionError as e:
        print(status+"FAIL")
        failed += 1
        print("  Expected:")
        print(expected)
        print("  Got:")
        print(result)
        if len(expected) != len(result):
            print(f"  Lengths: expected {len(expected)}, got {len(result)}")
        break
else:
    print('\nAll tests succesfull!')
print(f"\nSummary: {passed} passed, {failed} failed out of 80 tests.")

Running test cases...

Test 80.....PASS
All tests succesfull!

Summary: 80 passed, 0 failed out of 80 tests.


## Task 2 (50 pts). Analyze Subscription Conversion 

### Task Description

Table: `UserActivity`

| Column Name      | Type    | 
|------------------|---------|
| user_id          | int     |
| activity_date    | date    |
| activity_type    | varchar |
| activity_duration| int     |

(user_id, activity_date, activity_type) is the unique key for this table.\
activity_type is one of ('free_trial', 'paid', 'cancelled').\
activity_duration is the number of minutes the user spent on the platform that day.\
Each row represents a user's activity on a specific date.\
A subscription service wants to analyze user behavior patterns. The company offers a 7-day free trial, after which users can subscribe to a paid plan or cancel. Write a solution to:

Find users who converted from free trial to paid subscription\
Calculate each user's average daily activity duration during their free trial period (rounded to 2 decimal places)\
Calculate each user's average daily activity duration during their paid subscription period (rounded to 2 decimal places)\
Return the result table ordered by user_id in ascending order.

The result format is in the following example. Store your solution in the `sol_2` function

 

Example:

Input:

UserActivity table:

| user_id | activity_date | activity_type | activity_duration |
|---------|---------------|---------------|-------------------|
| 1       | 2023-01-01    | free_trial    | 45                |
| 1       | 2023-01-02    | free_trial    | 30                |
| 1       | 2023-01-05    | free_trial    | 60                |
| 1       | 2023-01-10    | paid          | 75                |
| 1       | 2023-01-12    | paid          | 90                |
| 1       | 2023-01-15    | paid          | 65                |
| 2       | 2023-02-01    | free_trial    | 55                |
| 2       | 2023-02-03    | free_trial    | 25                |
| 2       | 2023-02-07    | free_trial    | 50                |
| 2       | 2023-02-10    | cancelled     | 0                 |
| 3       | 2023-03-05    | free_trial    | 70                |
| 3       | 2023-03-06    | free_trial    | 60                |
| 3       | 2023-03-08    | free_trial    | 80                |
| 3       | 2023-03-12    | paid          | 50                |
| 3       | 2023-03-15    | paid          | 55                |
| 3       | 2023-03-20    | paid          | 85                |
| 4       | 2023-04-01    | free_trial    | 40                |
| 4       | 2023-04-03    | free_trial    | 35                |
| 4       | 2023-04-05    | paid          | 45                |
| 4       | 2023-04-07    | cancelled     | 0                 |

Output:


| user_id | trial_avg_duration | paid_avg_duration |
|---------|--------------------|-------------------|
| 1       | 45.00              | 76.67             |
| 3       | 70.00              | 63.33             |
| 4       | 37.50              | 45.00             |

Explanation:

User 1:
Had 3 days of free trial with durations of 45, 30, and 60 minutes.\
Average trial duration: (45 + 30 + 60) / 3 = 45.00 minutes.\
Had 3 days of paid subscription with durations of 75, 90, and 65 minutes.\
Average paid duration: (75 + 90 + 65) / 3 = 76.67 minutes.

User 2:
Had 3 days of free trial with durations of 55, 25, and 50 minutes.\
Average trial duration: (55 + 25 + 50) / 3 = 43.33 minutes.\
Did not convert to a paid subscription (only had free_trial and cancelled activities).\
Not included in the output because they didn't convert to paid.

User 3:
Had 3 days of free trial with durations of 70, 60, and 80 minutes.\
Average trial duration: (70 + 60 + 80) / 3 = 70.00 minutes.\
Had 3 days of paid subscription with durations of 50, 55, and 85 minutes.\
Average paid duration: (50 + 55 + 85) / 3 = 63.33 minutes.

User 4:
Had 2 days of free trial with durations of 40 and 35 minutes.\
Average trial duration: (40 + 35) / 2 = 37.50 minutes.\
Had 1 day of paid subscription with duration of 45 minutes before cancelling.\
Average paid duration: 45.00 minutes.\
The result table only includes users who converted from free trial to paid subscription (users 1, 3, and 4), and is ordered by user_id in ascending order.

### Solution

In [18]:
def sol_2(user_activity):
    if user_activity.empty:
        return pd.DataFrame(columns=['user_id', 'trial_avg_duration', 'paid_avg_duration']).astype({
            'trial_avg_duration': 'float', 'paid_avg_duration': 'float'
        })

    # Ensure activity_duration is numeric
    df = user_activity.copy()
    df['activity_duration'] = pd.to_numeric(df['activity_duration'], errors='coerce')

    free = df[df['activity_type'] == 'free_trial']
    paid = df[df['activity_type'] == 'paid']

    free_avg = free.groupby('user_id')['activity_duration'].mean().round(2).reset_index(name='trial_avg_duration')
    paid_avg = paid.groupby('user_id')['activity_duration'].mean().round(2).reset_index(name='paid_avg_duration')

    converted = free_avg.merge(paid_avg, on='user_id', how='inner')
    result = converted.sort_values('user_id').reset_index(drop=True)

    # Ensure columns are float (they should be, but just in case)
    result['trial_avg_duration'] = result['trial_avg_duration'].astype(float)
    result['paid_avg_duration'] = result['paid_avg_duration'].astype(float)

    return result

### Tests

Use the code block below to test your solutions

In [19]:
from pandas.testing import assert_frame_equal

print(f"Running test cases...\n")
passed = 0
failed = 0
for i in range(1, 94):
    status = f"\rTest {i}....."
    expected = pd.read_csv(fr'tests_task_2\exp_{i}.csv', index_col=0)
    usr_activity = pd.read_csv(fr'tests_task_2\usr_ac_{i}.csv', index_col=0)
    result = sol_2(usr_activity)
    try:
        assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
        print(status+"PASS", end='')
        passed += 1
    except AssertionError as e:
        print(status+"FAIL")
        failed += 1
        print("  Expected:")
        print(expected)
        print("  Got:")
        print(result)
        if len(expected) != len(result):
            print(f"  Lengths: expected {len(expected)}, got {len(result)}")
        break
else:
    print('\nAll tests succesfull!')
    
print(f"\nSummary: {passed} passed, {failed} failed out of 93 tests.")

Running test cases...

Test 93.....PASS
All tests succesfull!

Summary: 93 passed, 0 failed out of 80 tests.
